# PoliMillionaire Game Tests

Run selected local models one after another.

## Setup

In [1]:
import gc
import importlib.util
import os
import sys
import time
from datetime import datetime, timezone
from pathlib import Path


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "src" / "polimillionaire" / "__init__.py").exists():
            return candidate
    raise RuntimeError("Open this notebook from inside the project folder.")


REPO_ROOT = find_repo_root(Path.cwd().resolve())
for path in [REPO_ROOT / "src", REPO_ROOT]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ.setdefault("HF_HOME", str(REPO_ROOT / "data" / "hf_home"))
os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(REPO_ROOT / "data" / "hf_home" / "hub"))
print("Repo:", REPO_ROOT)
print("HF cache:", os.environ["HF_HOME"])

Repo: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire
HF cache: C:\Users\sjuan\Desktop\NLP\project\nlp-polimillionaire\data\hf_home


## Settings

Set `run=True` for the models you want to test.

In [2]:
RUN_API_CHECK = True
RUN_LIVE_GAME = True
RUN_OFFLINE_BENCHMARK = True
PROMPT_FOR_CREDENTIALS = False

API_URL = "http://131.175.15.22:51111/"
COMPETITION_IDS = [0, 1, 2, 3, 4, 5]
SAFE_DELAY_SECONDS = 2.0
ANSWER_TIMEOUT_SECONDS = 25.0

MODELS_TO_RUN = [
    {"label": "Gemma 4 E2B", "kind": "gemma", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E4B", "kind": "gemma", "model_id": "google/gemma-4-E4B-it", "run": False},
    {"label": "Qwen3.5 2B Thinking", "kind": "qwen_thinking", "model_id": "Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B Council", "kind": "gemma_council", "model_id": "google/gemma-4-E2B-it", "votes": 3, "run": False},
    {"label": "Gemma + Qwen Mixed Council", "kind": "mixed_council", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B (4-bit)", "kind": "gemma_quantized", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma + Qwen Mixed Council (4-bit)", "kind": "mixed_quantized", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": False},
    {"label": "Gemma 4 E2B LangChain Agent (4-bit)", "kind": "langchain_agentic", "model_id": "google/gemma-4-E2B-it", "run": False},
    {"label": "Gemma 4 E2B + Routed RAG", "kind": "rag", "model_id": "google/gemma-4-E2B-it", "run": True},
    {"label": "Gemma + Qwen Mixed Council + Routed RAG (4-bit)", "kind": "mixed_quantized_rag", "model_id": "google/gemma-4-E2B-it + Qwen/Qwen3.5-2B", "run": True}
]

print("API check:", RUN_API_CHECK, "Live game:", RUN_LIVE_GAME, "Benchmark:", RUN_OFFLINE_BENCHMARK)
print("Competition IDs:", COMPETITION_IDS)
for item in MODELS_TO_RUN:
    print("-", item["label"], "run=", item["run"])

API check: True Live game: True Benchmark: True
Competition IDs: [0, 1, 2, 3, 4, 5]
- Gemma 4 E2B run= False
- Gemma 4 E4B run= False
- Qwen3.5 2B Thinking run= False
- Gemma 4 E2B Council run= False
- Gemma + Qwen Mixed Council run= False
- Gemma 4 E2B (4-bit) run= False
- Gemma + Qwen Mixed Council (4-bit) run= False
- Gemma 4 E2B LangChain Agent (4-bit) run= False
- Gemma 4 E2B + Routed RAG run= True
- Gemma + Qwen Mixed Council + Routed RAG (4-bit) run= True


## Login

In [3]:
import getpass

from dotenv import load_dotenv
from millionaire_client import AuthenticationError, MillionaireClient

# Load environment variables from .env file
try:
    load_dotenv()
except Exception as e:
    print(f"Error loading .env file: {e}"   )

USERNAME = os.environ.get("POLIMILLIONAIRE_USERNAME")
PASSWORD = os.environ.get("POLIMILLIONAIRE_PASSWORD")
try:
    from google.colab import userdata
    USERNAME = USERNAME or userdata.get("POLIMILLIONAIRE_USERNAME")
    PASSWORD = PASSWORD or userdata.get("POLIMILLIONAIRE_PASSWORD")
except Exception:
    pass
if PROMPT_FOR_CREDENTIALS and not USERNAME:
    USERNAME = input("PoliMillionaire username: ").strip()
if PROMPT_FOR_CREDENTIALS and not PASSWORD:
    PASSWORD = getpass.getpass("PoliMillionaire password: ")
print("Username configured:", bool(USERNAME))
print("Password configured:", bool(PASSWORD))

Username configured: True
Password configured: True


## Competitions

In [4]:
client = None
if RUN_API_CHECK:
    if not USERNAME or not PASSWORD:
        raise RuntimeError("Set credentials first.")
    try:
        client = MillionaireClient(API_URL)
        user = client.login(USERNAME, PASSWORD)
        print("Logged in as", user.username)
        for competition in client.competitions.list_all():
            print(competition.id, competition.name, competition.max_levels)
    except AuthenticationError as exc:
        client = None
        print("Login failed:", exc)
else:
    print("API check skipped.")

Logged in as sanchez
0 Entertainment 15
1 Ancient History and Politics 15
2 Science and Nature 15
3 Maths 15
4 Philosophy and Psychology 15
5 News 15


## Run Selected Models

The 4-bit option follows the class quantization recipe and needs `bitsandbytes`.

In [5]:
from polimillionaire.runner import GameRunner, RunLogger, load_jsonl, summarize_attempts
from polimillionaire.strategies import CouncilStrategy, GemmaLLM, GemmaLLMConfig, GemmaStrategy, QwenLLM, QwenLLMConfig, QwenStrategy, LangChainAgenticStrategy, HeuristicStrategy, RAGStrategy, RAGConfig, RoutedStrategy
from polimillionaire.types import AnswerOption, Question

warmup_question = Question(1, "What is 2 + 2?", [AnswerOption(0, "3"), AnswerOption(1, "4"), AnswerOption(2, "5"), AnswerOption(3, "22")])
rag_warmup_question = Question(4, "In which year was A Haunted House 2 released?", [AnswerOption(0, "2012"), AnswerOption(1, "2015"), AnswerOption(2, "2014"), AnswerOption(3, "2013")])
BENCHMARK_SET = [
    (warmup_question, 1),
    (Question(2, "Which song was NOT written by Bob Dylan?", [AnswerOption(0, "Like a Rolling Stone"), AnswerOption(1, "Blowin' in the Wind"), AnswerOption(2, "The Times They Are A-Changin'"), AnswerOption(3, "Hound Dog")]), 3),
    (Question(3, "What was Whitney Houston's debut album?", [AnswerOption(0, "Whitney Houston"), AnswerOption(1, "Just Whitney"), AnswerOption(2, "I'm Your Baby Tonight"), AnswerOption(3, "Whitney")]), 0),
]


def clear_model_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.ipc_collect()
    except Exception as exc:
        print("Cleanup warning:", exc)


def mixed_council(quantized=False):
    if quantized and importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For the 4-bit council, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=quantized, max_new_tokens=32, do_sample=True, seed=42, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=quantized, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=6.0, timeout_seconds=120.0))
    return CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only" if quantized else "any_option", rejected_judge_fallback="primary_candidate" if quantized else "confidence_weighted", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)


def mixed_quantized_routed_rag():
    if importlib.util.find_spec("bitsandbytes") is None:
        raise RuntimeError("For 4-bit mixed routed RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
    gemma = GemmaLLM(GemmaLLMConfig(model_id="google/gemma-4-E2B-it", quantize_4bit=True, max_new_tokens=32, temperature=0.0, do_sample=False, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    qwen = QwenLLM(QwenLLMConfig(model_id="Qwen/Qwen3.5-2B", quantize_4bit=True, max_new_tokens=32, temperature=0.7, top_p=0.8, top_k=20, do_sample=True, seed=43, enable_thinking=False, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    direct = GemmaStrategy(llm=gemma)
    council = CouncilStrategy(candidate_llms=[gemma, qwen], judge_llm=gemma, judge_scope="candidate_only", rejected_judge_fallback="primary_candidate", base_seed=200, candidate_max_new_tokens=32, judge_max_new_tokens=8, max_time_per_call=6.0)
    rag = RAGStrategy(llm=gemma, retrieval_config=RAGConfig(num_extra_queries=0), fallback_strategy=council)
    return RoutedStrategy(direct_strategy=direct, rag_strategy=rag, fallback_strategy=council, low_confidence_strategy=council, rag_min_confidence=0.65)


def has_rag_strategy(strategy):
    if isinstance(strategy, RAGStrategy):
        return True
    if isinstance(strategy, RoutedStrategy):
        return has_rag_strategy(strategy.rag_strategy)
    return False


def strategy_devices(strategy):
    devices = []

    def collect(item):
        if item is None:
            return
        if isinstance(item, CouncilStrategy):
            for llm in item.candidate_llms + [item.judge_llm]:
                devices.append(getattr(llm, "device_summary", "unknown"))
            return
        if isinstance(item, RoutedStrategy):
            collect(item.direct_strategy)
            collect(item.rag_strategy)
            collect(item.fallback_strategy)
            return
        if hasattr(item, "llm"):
            devices.append(getattr(item.llm, "device_summary", "unknown"))

    collect(strategy)
    return list(dict.fromkeys(devices))


def make_strategy(item):
    if item["kind"] == "gemma_quantized":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit Gemma, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], quantize_4bit=True, max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "mixed_quantized":
        return mixed_council(quantized=True)
    if item["kind"] == "mixed_quantized_rag":
        return mixed_quantized_routed_rag()
    if item["kind"] == "langchain_agentic":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit LangChain Agent, run `%pip install -U bitsandbytes`.")
        
        agent_config = GemmaLLMConfig(
            model_id=item["model_id"], 
            quantize_4bit=True, 
            max_new_tokens=128,
            temperature=0.0, 
            do_sample=False, 
            generation_max_time_seconds=30.0, 
            timeout_seconds=120.0
        )
        base_llm = GemmaLLM(config=agent_config)
        return LangChainAgenticStrategy(raw_llm=base_llm, fallback_strategy=HeuristicStrategy())
    if item["kind"] == "mixed_council":
        return mixed_council()
    if item["kind"] == "gemma_council":
        llm = GemmaLLM(GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=48, do_sample=True, seed=42, generation_max_time_seconds=4.5, timeout_seconds=120.0))
        return CouncilStrategy(llm=llm, num_votes=item.get("votes", 3), base_seed=100, candidate_max_new_tokens=48, judge_max_new_tokens=8, max_time_per_call=4.5)
    if item["kind"] == "qwen_thinking":
        return QwenStrategy(model_config=QwenLLMConfig(model_id=item["model_id"], max_new_tokens=128, temperature=1.0, top_p=0.95, top_k=20, do_sample=True, seed=42, enable_thinking=True, generation_max_time_seconds=18.0, timeout_seconds=120.0))
    if item["kind"] == "rag":
        if importlib.util.find_spec("bitsandbytes") is None:
            raise RuntimeError("For 4-bit RAG, run `%pip install -U bitsandbytes`, restart the kernel, and rerun setup.")
        base_llm = GemmaLLM(GemmaLLMConfig(
            model_id=item["model_id"],
            quantize_4bit=True,
            max_new_tokens=32,
            temperature=0.0,
            do_sample=False,
            generation_max_time_seconds=18.0,
            timeout_seconds=120.0,
        ))
        direct_strategy = GemmaStrategy(llm=base_llm)
        rag_strategy = RAGStrategy(
            llm=base_llm,
            retrieval_config=RAGConfig(num_extra_queries=0),  # disable expansion for speed
            fallback_strategy=HeuristicStrategy(),
        )
        return RoutedStrategy(direct_strategy=direct_strategy, rag_strategy=rag_strategy, fallback_strategy=direct_strategy)
    return GemmaStrategy(model_config=GemmaLLMConfig(model_id=item["model_id"], max_new_tokens=8, temperature=0.0, do_sample=False, num_beams=1, seed=42, generation_max_time_seconds=18.0, timeout_seconds=120.0))


def clean_name(label):
    return "_".join(label.lower().replace("+", " ").replace("(", " ").replace(")", " ").split())


benchmark_results = []


def benchmark(strategy, label):
    rows = []
    for question, gold_id in BENCHMARK_SET:
        started = time.monotonic()
        prediction = strategy.answer(question)
        seconds = time.monotonic() - started
        correct = prediction.option_id == gold_id
        rows.append((correct, seconds, bool(prediction.metadata.get("fallback"))))
        gold = question.require_option(gold_id)
        print("\nQ:", question.text)
        print("predicted:", prediction.option_id, prediction.answer_text)
        print("gold:", gold.id, gold.text, "correct:", correct, "seconds:", round(seconds, 2))
        print("decision:", prediction.metadata.get("decision_method"), "route:", prediction.metadata.get("route"), "fallback:", prediction.metadata.get("fallback"))
        for index, vote in enumerate(prediction.metadata.get("votes") or [], start=1):
            print("vote", index, vote.get("model_name"), "->", vote.get("option_id"), "confidence:", vote.get("confidence"), "reason:", vote.get("reasoning"))
        if prediction.metadata.get("judge_raw_text") is not None:
            print("judge raw:", prediction.metadata.get("judge_raw_text"), "scope:", prediction.metadata.get("judge_scope"))
        for source in prediction.metadata.get("evidence_sources") or []:
            print("evidence:", source.get("title"), source.get("url"))
    accuracy = sum(correct for correct, _, _ in rows) / len(rows)
    max_seconds = max(seconds for _, seconds, _ in rows)
    summary = {"model": label, "accuracy": accuracy, "avg_seconds": round(sum(seconds for _, seconds, _ in rows) / len(rows), 2), "max_seconds": round(max_seconds, 2), "fallbacks": sum(fallback for _, _, fallback in rows)}
    benchmark_results.append(summary)
    print("Benchmark summary:", summary)
    return accuracy >= 0.60 and max_seconds <= 20.0 and not any(fallback for _, _, fallback in rows)


if RUN_LIVE_GAME and (not USERNAME or not PASSWORD):
    raise RuntimeError("Set credentials first.")
if RUN_LIVE_GAME and client is None:
    client = MillionaireClient(API_URL)
    client.login(USERNAME, PASSWORD)

run_results = []
for item in MODELS_TO_RUN:
    if not item.get("run", False):
        print("Skipped", item["label"])
        continue
    strategy = None
    try:
        print("\nModel:", item["label"])
        strategy = make_strategy(item)
        started = time.monotonic()
        warmup = strategy.answer(warmup_question)
        load_and_warmup_seconds = time.monotonic() - started
        print("warmup option:", warmup.option_id, warmup.answer_text, "fallback:", warmup.metadata.get("fallback"), "route:", warmup.metadata.get("route"), "load + answer seconds:", round(load_and_warmup_seconds, 2))
        if warmup.metadata.get("fallback") or warmup.option_id != 1:
            raise RuntimeError("Warm-up failed. No live game started.")
        if item["kind"] in {"gemma_quantized", "mixed_quantized", "mixed_quantized_rag"}:
            devices = strategy_devices(strategy)
            print("devices:", devices)
            if any(token in " ".join(devices).lower() for token in ("'cpu'", "'disk'", "meta")):
                raise RuntimeError("Quantized model was offloaded. No live game started.")
            started = time.monotonic()
            speed_check = strategy.answer(warmup_question)
            answer_seconds = time.monotonic() - started
            print("loaded-model speed check:", round(answer_seconds, 2), "seconds")
            if speed_check.metadata.get("fallback") or speed_check.option_id != 1:
                raise RuntimeError("Quantized model speed check failed. No live game started.")
            if answer_seconds > 20.0:
                raise RuntimeError("Quantized model loaded answer is over 20 seconds. No live game started.")
        if has_rag_strategy(strategy):
            started = time.monotonic()
            rag_warmup = strategy.answer(rag_warmup_question)
            rag_warmup_seconds = time.monotonic() - started
            print("rag warmup option:", rag_warmup.option_id, rag_warmup.answer_text, "fallback:", rag_warmup.metadata.get("fallback"), "route:", rag_warmup.metadata.get("route"), "seconds:", round(rag_warmup_seconds, 2))
            if rag_warmup.metadata.get("fallback") or rag_warmup.option_id != 2:
                raise RuntimeError("RAG warm-up failed. No live game started.")
        benchmark_ok = benchmark(strategy, item["label"]) if RUN_OFFLINE_BENCHMARK else True
        if item["kind"] in {"gemma_quantized", "mixed_quantized", "mixed_quantized_rag"} and RUN_LIVE_GAME and not RUN_OFFLINE_BENCHMARK:
            raise RuntimeError("Set RUN_OFFLINE_BENCHMARK=True before a live 4-bit run.")
        if not benchmark_ok and RUN_LIVE_GAME:
            raise RuntimeError("Benchmark guard failed. No live game started.")
        if not benchmark_ok:
            print("Benchmark below live threshold; kept for offline comparison.")
        result = {"label": item["label"], "warmup_option": warmup.option_id, "benchmark_ok": benchmark_ok}
        if RUN_LIVE_GAME:
            live_runs = []
            for competition_id in COMPETITION_IDS:
                run_id = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
                log_path = REPO_ROOT / "results" / "runs" / f"{clean_name(item['label'])}_competition_{competition_id}_{run_id}.jsonl"
                game = GameRunner(client, SAFE_DELAY_SECONDS, ANSWER_TIMEOUT_SECONDS, RunLogger(log_path)).play(competition_id, strategy)
                summary = summarize_attempts(load_jsonl(log_path)) if log_path.exists() else {}
                live_run = {"competition_id": competition_id, "earned_amount": game.earned_amount, "log_path": str(log_path), "summary": summary}
                live_runs.append(live_run)
                print("Competition:", competition_id, "Earned amount:", game.earned_amount, "Log path:", log_path)
                print("Summary:", summary)
            result.update({
                "live_runs": live_runs,
                "log_paths": [run["log_path"] for run in live_runs],
                "earned_amount_total": sum((run.get("earned_amount") or 0) for run in live_runs),
            })
        run_results.append(result)
    finally:
        del strategy
        clear_model_memory()
print("\nComparison:")
for summary in benchmark_results:
    print(summary)
run_results

Skipped Gemma 4 E2B
Skipped Gemma 4 E4B
Skipped Qwen3.5 2B Thinking
Skipped Gemma 4 E2B Council
Skipped Gemma + Qwen Mixed Council
Skipped Gemma 4 E2B (4-bit)
Skipped Gemma + Qwen Mixed Council (4-bit)
Skipped Gemma 4 E2B LangChain Agent (4-bit)

Model: Gemma 4 E2B + Routed RAG


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

warmup option: 1 4 fallback: False route: direct load + answer seconds: 21.34


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

rag warmup option: 2 2014 fallback: False route: rag seconds: 24.56

Q: What is 2 + 2?
predicted: 1 4
gold: 1 4 correct: True seconds: 0.64
decision: None route: direct fallback: False

Q: Which song was NOT written by Bob Dylan?
predicted: 3 Hound Dog
gold: 3 Hound Dog correct: True seconds: 0.62
decision: None route: direct fallback: False

Q: What was Whitney Houston's debut album?
predicted: 0 Whitney Houston
gold: 0 Whitney Houston correct: True seconds: 10.3
decision: None route: rag fallback: False
evidence: Whitney Houston (album) - Wikipedia https://en.wikipedia.org/wiki/Whitney_Houston_(album)
evidence: Whitney Houston (Album) | Whitney Houston Wiki | Fandom https://whitney-houston.fandom.com/wiki/Whitney_Houston_(Album)
evidence: The List of Whitney Houston Albums in Order of Release https://albumsinorder.com/whitney-houston-albums-in-order/
Benchmark summary: {'model': 'Gemma 4 E2B + Routed RAG', 'accuracy': 1.0, 'avg_seconds': 3.85, 'max_seconds': 10.3, 'fallbacks': 0}
Com

[transformers] Current model requires 6178 bytes of buffer for offloaded layers, which seems does not fit any GPU's remaining memory. If you are experiencing a OOM later, please consider using offload_buffers=True.


Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

warmup option: 0 3 fallback: True route: direct load + answer seconds: 106.91


RuntimeError: Warm-up failed. No live game started.

## Results

In [ ]:
from pathlib import Path
from polimillionaire.runner import load_jsonl, summarize_attempts

print("Benchmark results:")
for row in benchmark_results:
    print(row)

live_logs = []
for item in run_results:
    for log_path in item.get("log_paths", []):
        live_logs.append(Path(log_path))
    if item.get("log_path"):
        live_logs.append(Path(item["log_path"]))
if live_logs:
    print("\nLive runs from this execution:")
    for latest in live_logs:
        print(latest.name)
        print(summarize_attempts(load_jsonl(latest)))
else:
    print("\nNo live game was run in this notebook execution.")
    old_logs = sorted((REPO_ROOT / "results" / "runs").glob("*.jsonl"), key=lambda path: path.stat().st_mtime)
    if old_logs:
        print("Most recent saved live log, not from this run:", old_logs[-1].name)


Benchmark results:
{'model': 'Gemma + Qwen Mixed Council + Routed RAG (4-bit)', 'accuracy': 1.0, 'avg_seconds': 3.67, 'max_seconds': 9.75, 'fallbacks': 0}

No live game was run in this notebook execution.
Most recent saved live log, not from this run: gemma_4_e2b_routed_rag_20260528_112325.jsonl
